# Train / Val Split — Great Barrier Reef Starfish Detection

Produces `data/splits.csv`, the canonical `image_id -> split` mapping used by
all downstream code (`dataset.py`, `model.py`).

**Design:** split is done per-video, at the sequence level (not a literal
whole-video holdout). With only 3 videos and very different empty-frame rates
(68% / 74.5% / 92%), holding out one whole video for val would be unfair —
val would be either near-empty or annotation-heavy relative to train.
Splitting each video's sequences independently (then combining) guarantees
every video is represented in both train and val, keeps each video's
empty/non-empty ratio consistent across the split, and still prevents
near-duplicate adjacent frames from leaking across train/val (a sequence is
always kept fully on one side).

**Known limitation:** video 2 has only 4 sequences, so its val split can't be
finely balanced by empty-rate — see the fairness report below.

**Usage in `model.py`:**
```python
train_csv = pd.read_csv('/content/data/train.csv')
train_csv['annotations'] = train_csv['annotations'].apply(ast.literal_eval)
splits = pd.read_csv('data/splits.csv')
full = train_csv.merge(splits[['image_id', 'split']], on='image_id')
train_data = full[full['split'] == 'train']
val_data = full[full['split'] == 'val']
```
Image path convention: `train_images/video_{video_id}/{video_frame}.jpg`,
where `video_id, video_frame = image_id.split('-')`.

In [1]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [2]:
csv_path = '/content/data/train.csv'

if os.path.exists(csv_path):
    print(f'Found existing {csv_path}, skipping download')
else:
    os.makedirs('/content/data', exist_ok=True)
    os.system('kaggle competitions download -c tensorflow-great-barrier-reef -f train.csv -p /content/data')
    os.system('unzip -o /content/data/train.csv.zip -d /content/data')
    print(f'Downloaded and unzipped {csv_path}')

Downloaded and unzipped /content/data/train.csv


Load data and create sequence groups

In [3]:
import ast
import pandas as pd

df = pd.read_csv(csv_path)
df['annotations'] = df['annotations'].apply(ast.literal_eval)
df['is_empty'] = df['annotations'].apply(len) == 0

# sequence IDs repeat across videos, so build a composite key
df['seq_group'] = df['video_id'].astype(str) + '_' + df['sequence'].astype(str)

print(f"Total frames: {len(df)}")
print(f"Total sequences: {df['seq_group'].nunique()}")
print(f"Overall empty frames: {df['is_empty'].mean() * 100:.1f}%")

Total frames: 23501
Total sequences: 20
Overall empty frames: 79.1%


Find the best sequence-level split

In [4]:
from sklearn.model_selection import GroupShuffleSplit

VAL_SIZE = 0.20
N_CANDIDATES = 500
RANDOM_STATE = 42

def find_best_video_split(video_df,val_size=0.20,n_candidates=500,random_state=42):
    """
    Try multiple sequence-level train/validation splits for one video
    and choose the split whose validation distribution is closest
    to the full video's distribution.
    Complete sequences always stay together.
    """
    target_empty_rate = video_df['is_empty'].mean()

    splitter = GroupShuffleSplit(
        n_splits=n_candidates,
        test_size=val_size,
        random_state=random_state
    )

    best_score = float('inf')
    best_train = None
    best_val = None

    for train_idx, val_idx in splitter.split(video_df,groups=video_df['seq_group']):
        candidate_train = video_df.iloc[train_idx]
        candidate_val = video_df.iloc[val_idx]

        # How close is validation size to the requested 20%?
        actual_val_fraction = len(candidate_val) / len(video_df)
        size_difference = abs(actual_val_fraction - val_size)

        # How close is validation empty-frame rate to this video's overall empty-frame rate?
        val_empty_rate = candidate_val['is_empty'].mean()
        empty_rate_difference = abs(val_empty_rate - target_empty_rate)

        # Combined score:smaller = better
        score = size_difference + empty_rate_difference

        if score < best_score:
            best_score = score
            best_train = candidate_train.copy()
            best_val = candidate_val.copy()

    return best_train, best_val, best_score

Select the best split independently for each video

In [5]:
train_parts, val_parts = [], []

for video_id in sorted(df['video_id'].unique()):
    video_df = df[df['video_id'] == video_id].reset_index(drop=True)
    train_part, val_part, score = find_best_video_split(
        video_df,
        val_size=VAL_SIZE,
        n_candidates=N_CANDIDATES,
        random_state=RANDOM_STATE
    )
    train_parts.append(train_part)
    val_parts.append(val_part)
    print(f"Video {video_id}: ",f"train={len(train_part)}",f"val={len(val_part)}",f"score={score:.4f}")

train_df = pd.concat(train_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)

Video 0:  train=5430 val=1278 score=0.0097
Video 1:  train=6978 val=1254 score=0.1432
Video 2:  train=7036 val=1525 score=0.0819


Leakage and integrity checks

In [6]:
assert set(train_df['seq_group']).isdisjoint(set(val_df['seq_group']))
assert len(train_df) + len(val_df) == len(df)
assert set(train_df['image_id']).isdisjoint(set(val_df['image_id']))
assert (set(train_df['image_id']) | set(val_df['image_id'])== set(df['image_id']))

print("✓ No sequence leakage")
print("✓ No image leakage")
print("✓ All frames accounted for")

✓ No sequence leakage
✓ No image leakage
✓ All frames accounted for


Examine the resulting split

In [7]:
print(f"Train: {len(train_df)} frames",f"{train_df['is_empty'].mean() * 100:.1f}% empty")
print(f"Val:   {len(val_df)} frames",f"{val_df['is_empty'].mean() * 100:.1f}% empty")
print("\nPer-video frame counts:")
print(pd.concat([train_df['video_id'].value_counts().rename('train'),val_df['video_id'].value_counts().rename('val')],
        axis=1).sort_index())
print("\nPer-video empty %:")
empty_report = pd.concat([(
            train_df.groupby('video_id')['is_empty'].mean() * 100
        ).rename('train'),
        (
            val_df.groupby('video_id')['is_empty'].mean() * 100
        ).rename('val'),
        (
            df.groupby('video_id')['is_empty'].mean() * 100
        ).rename('full')],axis=1).round(1)
print(empty_report)

Train: 19444 frames 78.0% empty
Val:   4057 frames 84.3% empty

Per-video frame counts:
          train   val
video_id             
0          5430  1278
1          6978  1254
2          7036  1525

Per-video empty %:
          train   val  full
video_id                   
0          68.0  68.1  68.1
1          72.8  84.1  74.5
2          90.8  98.1  92.1


Save splits.csv

In [8]:
train_df['split'] = 'train'
val_df['split'] = 'val'
full_df = pd.concat([train_df, val_df]).reset_index(drop=True)
full_df = full_df[['image_id', 'video_id', 'sequence', 'split']]

os.makedirs('data', exist_ok=True)
full_df.to_csv('data/splits.csv', index=False)
print(full_df['split'].value_counts())

split
train    19444
val       4057
Name: count, dtype: int64
